In [5]:
from pathlib import Path
from comptest import libtiff, imagemagick
import subprocess

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
COMPRESSION_TESTS_DIR = Path(
    "/dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests"
)
assert COMPRESSION_TESTS_DIR.is_dir()

In [7]:
!tiffcp --help

tiffcp: invalid option -- '-'
LIBTIFF, Version 4.7.2
Copyright (c) 1988-1996 Sam Leffler
Copyright (c) 1991-1996 Silicon Graphics, Inc.

Copy, convert, or combine TIFF files

usage: tiffcp [options] input... output
where options are:
 -a              append to output instead of overwriting
 -o offset       set initial directory offset
 -p contig       pack samples contiguously (e.g. RGBRGB...)
 -p separate     store samples separately (e.g. RRR...GGG...BBB...)
 -s              write output in strips
 -t              write output in tiles
 -x              force the merged tiff pages in sequence
 -8              write BigTIFF instead of default ClassicTIFF
 -B              write big-endian instead of native byte order
 -L              write little-endian instead of native byte order
 -M              disable use of memory-mapped files
 -C              disable strip chopping
 -i              ignore read errors
 -b file[,#]     bias (dark) monochrome image to be subtracted from all others
 

In [ ]:
input_file = COMPRESSION_TESTS_DIR / "uncompressed.tif"
assert input_file.is_file()

TESTS = {
    # ============================== N-BIT LOSSLESS ==============================
    "baseline_none": {
        "libtiff": ["-c", "none"],
        "imagemagick": ["-compress", "None"],
        "notes": ["lossless"],
    },
    # --- LZW ---
    "lzw_pred1": {
        "libtiff": ["-c", "lzw:1"],
        "imagemagick": ["-compress", "LZW", "-define", "tiff:predictor=1"],
        "notes": ["lossless"],
    },
    "lzw_pred2": {
        "libtiff": ["-c", "lzw:2"],
        "imagemagick": ["-compress", "LZW", "-define", "tiff:predictor=2"],
        "notes": ["lossless"],
    },
    # --- ZIP / DEFLATE ---
    "zip_pred1_p1": {
        "libtiff": ["-c", "zip:1:p1"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "10",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p3": {
        "libtiff": ["-c", "zip:1:p3"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "30",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p5": {
        "libtiff": ["-c", "zip:1:p5"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "50",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p6": {
        "libtiff": ["-c", "zip:1:p6"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p7": {
        "libtiff": ["-c", "zip:1:p7"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "70",
        ],
        "notes": ["lossless"],
    },
    "zip_pred1_p9": {
        "libtiff": ["-c", "zip:1:p9"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p1": {
        "libtiff": ["-c", "zip:2:p1"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "10",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p3": {
        "libtiff": ["-c", "zip:2:p3"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "30",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p5": {
        "libtiff": ["-c", "zip:2:p5"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "50",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p6": {
        "libtiff": ["-c", "zip:2:p6"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p7": {
        "libtiff": ["-c", "zip:2:p7"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "70",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p9": {
        "libtiff": ["-c", "zip:2:p9"],
        "imagemagick": [
            "-compress",
            "Zip",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    "zip_pred2_p9_s0_zlib": {
        "libtiff": ["-c", "zip:2:p9:s0"],
        "imagemagick": None,  # ImageMagick has no zlib/libdeflate subcodec switch
        "notes": ["lossless"],
    },
    "zip_pred2_p9_s1_libdeflate": {
        "libtiff": ["-c", "zip:2:p9:s1"],
        "imagemagick": None,  # ImageMagick has no zlib/libdeflate subcodec switch
        "notes": ["lossless"],
    },
    # --- LZMA ---
    "lzma_pred1_p0": {
        "libtiff": ["-c", "lzma:1:p0"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "0",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p2": {
        "libtiff": ["-c", "lzma:1:p2"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "20",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p4": {
        "libtiff": ["-c", "lzma:1:p4"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "40",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p6": {
        "libtiff": ["-c", "lzma:1:p6"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p8": {
        "libtiff": ["-c", "lzma:1:p8"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "80",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred1_p9": {
        "libtiff": ["-c", "lzma:1:p9"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p0": {
        "libtiff": ["-c", "lzma:2:p0"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "0",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p2": {
        "libtiff": ["-c", "lzma:2:p2"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "20",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p4": {
        "libtiff": ["-c", "lzma:2:p4"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "40",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p6": {
        "libtiff": ["-c", "lzma:2:p6"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "60",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p8": {
        "libtiff": ["-c", "lzma:2:p8"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "80",
        ],
        "notes": ["lossless"],
    },
    "lzma_pred2_p9": {
        "libtiff": ["-c", "lzma:2:p9"],
        "imagemagick": [
            "-compress",
            "LZMA",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "90",
        ],
        "notes": ["lossless"],
    },
    # --- ZSTD ---
    "zstd_pred1_p1": {
        "libtiff": ["-c", "zstd:1:p1"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "5",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p5": {
        "libtiff": ["-c", "zstd:1:p5"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "23",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p9": {
        "libtiff": ["-c", "zstd:1:p9"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "41",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p13": {
        "libtiff": ["-c", "zstd:1:p13"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "59",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p17": {
        "libtiff": ["-c", "zstd:1:p17"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "77",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p19": {
        "libtiff": ["-c", "zstd:1:p19"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "86",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred1_p22": {
        "libtiff": ["-c", "zstd:1:p22"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=1",
            "-quality",
            "100",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p1": {
        "libtiff": ["-c", "zstd:2:p1"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "5",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p5": {
        "libtiff": ["-c", "zstd:2:p5"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "23",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p9": {
        "libtiff": ["-c", "zstd:2:p9"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "41",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p13": {
        "libtiff": ["-c", "zstd:2:p13"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "59",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p17": {
        "libtiff": ["-c", "zstd:2:p17"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "77",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p19": {
        "libtiff": ["-c", "zstd:2:p19"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "86",
        ],
        "notes": ["lossless"],
    },
    "zstd_pred2_p22": {
        "libtiff": ["-c", "zstd:2:p22"],
        "imagemagick": [
            "-compress",
            "Zstd",
            "-define",
            "tiff:predictor=2",
            "-quality",
            "100",
        ],
        "notes": ["lossless"],
    },
    # --- LERC (z0 = lossless) ---
    "lerc_z0_s0_none": {
        "libtiff": ["-c", "lerc:0:s0"],
        "imagemagick": None,  # no LERC coder in ImageMagick
        "notes": ["lossless"],
    },
    "lerc_z0_s1_deflate_p1": {
        "libtiff": ["-c", "lerc:0:s1:p1"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s1_deflate_p5": {
        "libtiff": ["-c", "lerc:0:s1:p5"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s1_deflate_p9": {
        "libtiff": ["-c", "lerc:0:s1:p9"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p1": {
        "libtiff": ["-c", "lerc:0:s2:p1"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p9": {
        "libtiff": ["-c", "lerc:0:s2:p9"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p13": {
        "libtiff": ["-c", "lerc:0:s2:p13"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p17": {
        "libtiff": ["-c", "lerc:0:s2:p17"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    "lerc_z0_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:0:s2:p22"],
        "imagemagick": None,
        "notes": ["lossless"],
    },
    # --- PackBits ---
    "packbits": {
        "libtiff": ["-c", "packbits"],
        "imagemagick": ["-compress", "RLE"],
        "notes": ["lossless"],
    },
    # --- SGILOG ---
    "sgilog": {
        "libtiff": ["-c", "sgilog"],
        "imagemagick": ["-compress", "SGILog"],
        "notes": ["lossless", "designed-for-logluv-hdr"],
    },
    # ============================== N-BIT LOSSY (LERC z>0) ==============================
    "lerc_z0.5_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:0.5:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z1_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:1:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z2_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:2:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z5_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:5:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z10_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:10:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    "lerc_z20_s2_zstd_p22": {
        "libtiff": ["-c", "lerc:20:s2:p22"],
        "imagemagick": None,
        "notes": ["lossy"],
    },
    # ============================== 8-BIT LOSSLESS ==============================
    # libtiff: None -- tiffcp requires 8-bit source data already; can't downsample inline
    "webp_lossless_pred1_p50": {
        "libtiff": None,  # WEBP codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "50",
        ],
        "notes": ["lossless", "8bit"],
    },
    "webp_lossless_pred1_p75": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "75",
        ],
        "notes": ["lossless", "8bit"],
    },
    "webp_lossless_pred1_p100": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "WebP",
            "-define",
            "webp:lossless=true",
            "-quality",
            "100",
        ],
        "notes": ["lossless", "8bit"],
    },
    "g3_1d": {
        "libtiff": None,  # G3 is bilevel-only; source is 16-bit grayscale
        "imagemagick": ["-depth", "8", "-monochrome", "-compress", "Fax"],
        "notes": ["lossless", "1bit"],
    },
    "g3_2d": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-monochrome",
            "-compress",
            "Fax",
        ],  # 1D/2D not separately exposed
        "notes": ["lossless", "1bit"],
    },
    "g3_2d_fill": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-monochrome",
            "-compress",
            "Fax",
        ],  # fill/EOL byte-align not exposed
        "notes": ["lossless", "1bit"],
    },
    "g4": {
        "libtiff": None,  # G4 is bilevel-only; source is 16-bit grayscale
        "imagemagick": ["-depth", "8", "-monochrome", "-compress", "Group4"],
        "notes": ["lossless", "1bit"],
    },
    # ============================== 8-BIT LOSSY ==============================
    "webp_pred2_p25": {
        "libtiff": None,  # WEBP codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "25"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p50": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "50"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p65": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "65"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p75": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "75"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p90": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "90"],
        "notes": ["lossy", "8bit"],
    },
    "webp_pred2_p100": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "WebP", "-quality", "100"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q10": {
        "libtiff": None,  # JPEG codec in libtiff is 8-bit only; source is 16-bit
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "10"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q25": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "25"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q50": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "50"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q65": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "65"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q75": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "75"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q85": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "85"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q95": {
        "libtiff": None,
        "imagemagick": ["-depth", "8", "-compress", "JPEG", "-quality", "95"],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q50_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "50",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q75_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "75",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q85_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "85",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
    "jpeg_q95_rgb": {
        "libtiff": None,
        "imagemagick": [
            "-depth",
            "8",
            "-compress",
            "JPEG",
            "-quality",
            "95",
            "-colorspace",
            "RGB",
        ],
        "notes": ["lossy", "8bit"],
    },
}

OUTPUT_DIR = input_file.parent / "compressed"
OUTPUT_DIR.mkdir(exist_ok=True)
for name, setup in TESTS.items():
    if args := setup.get("libtiff"):
        output_path = OUTPUT_DIR / (name + "_libtiff.tif")
        print(f"LIBTIFF: Compressing {input_file} with args {args} -> {name}")
        try:
            libtiff.compress(input_file, output_path=output_path, args=args)
        except subprocess.CalledProcessError as e:
            print(f"LIBTIFF: Error compressing {input_file} with args {args}:")
            print(e.stdout.decode())
            print(e.stderr.decode())

    if args := setup.get("imagemagick"):
        output_path = OUTPUT_DIR / (name + "_imagemagick.tif")
        print(f"IMAGEMAGICK: Compressing {input_file} with args {args} -> {name}")
        try:
            imagemagick(input_file, output_path=output_path, args=args)
        except subprocess.CalledProcessError as e:
            print(f"IMAGEMAGICK: Error compressing {input_file} with args {args}:")
            print(e.stdout.decode())
            print(e.stderr.decode())

# tiff.compress(input_file, args=["-c", "lzw:3"])

LIBTIFF: Compressing /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests/uncompressed.tif with args ['-c', 'none'] -> baseline_none
IMAGEMAGICK: Compressing /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests/uncompressed.tif with args ['-compress', 'None'] -> baseline_none
LIBTIFF: Compressing /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests/uncompressed.tif with args ['-c', 'lzw:1'] -> lzw_pred1
IMAGEMAGICK: Compressing /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests/uncompressed.tif with args ['-compress', 'LZW', '-define', 'tiff:predictor=1'] -> lzw_pred1
LIBTIFF: Compressing /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests/uncompressed.tif with args ['-c', 'lzw:2'] -> lzw_pred2
IMAGEMAGICK: Compressing /dtu/3d-imaging-center/projects/2026_DANFIX_201/analysis/compression/compression_tests/uncompresse